# Data Cleaning and Normalization

This notebook prepares the raw vehicle listings for analysis and modeling. It exports "cars_cleaned" as the cleaned dataset.


In [22]:
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)


## 1. Load the raw dataset


In [23]:
df = pd.read_csv("cars.csv")
print("Initial shape:", df.shape)
display(df.head())


Initial shape: (4037, 10)


,brand,model,trim,year,mileage_km,city,price_toman,published_date,body_condition,ad_url
0,کوییک,دنده ای R,دنده ای,1401,98000,دماوند,1350000000,NaN,دارای خط و خش,https://bama.ir/car/detail-pyzvr7jb-quick-manu...
1,هایما,S5,6 سرعته اتوماتیک,1401,75000,رشت,4000000000,NaN,دارای خط و خش,https://bama.ir/car/detail-cqoddtxb-haima-s5-6...
2,ام وی ام,550,دنده ای,1394,206000,کرمانشاه,1430000000,NaN,دارای خط و خش,https://bama.ir/car/detail-bo7sghqw-mvm-550-mt...
3,فردا,T5,اتوماتیک,1403,84000,شاهرود,4000000000,NaN,دارای خط و خش,https://bama.ir/car/detail-hdys2m0i-farda-t5-1403
4,کی ام سی,K7,اتوماتیک,1402,44000,تهران,5900000000,NaN,دارای خط و خش,https://bama.ir/car/detail-8bdp9q3z-kmc-k7-1402


### We dont need "ad_url".
### also, published date is since "1 month ago" and due to our data quantity limitations, it wont be useful

## 2. Remove ad_url and some types of selling
### in our datasets I encounter to some cheats in selling such as "pre-sels or پیش فروش" and "transfer sale or حواله فروش" which are not useful

In [24]:
# URL is an identifier and is not a useful modeling feature.
df = df.drop(columns=["ad_url"], errors="ignore")

# Remove transfer / pre-sale listings because the target is used-car market price.
brand_text = df["brand"].astype("string")
excluded_mask = (
    brand_text.str.contains("حواله", na=False)
    | brand_text.str.contains(r"پیش\s?فروش", na=False)
)
print("Rows removed as transfer/pre-sale:", int(excluded_mask.sum()))
df = df.loc[~excluded_mask].copy()


Rows removed as transfer/pre-sale: 17


## 3. Normalize text fields

### do some typical "persian" text normalizations and also, create a trim as "بدون تیپ" because some cars have no trim!

In [25]:
def normalize_text(value):
    if pd.isna(value):
        return value
    value = str(value)
    value = value.replace("ي", "ی").replace("ى", "ی")
    value = value.replace("ك", "ک")
    value = value.replace("\u200c", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value

for col in ["brand", "model", "trim", "city", "body_condition"]:
    if col in df.columns:
        df[col] = df[col].map(normalize_text)

# Missing trim is represented explicitly.
if "trim" in df.columns:
    df["trim"] = df["trim"].fillna("بدون تیپ")


## 4. Normalize price and year

### converting price and mileage to numeric and also make all years in "SHAMSI" to have only one calendar.

In [26]:
# Raw target is converted to million Toman.
df["price_toman"] = pd.to_numeric(df["price_toman"], errors="coerce") * 1e-6

# Some records contain Gregorian years. Convert them to shamsi using
df["year"] = pd.to_numeric(df["year"], errors="coerce")
gregorian_mask = df["year"] > 1900
df.loc[gregorian_mask, "year"] = df.loc[gregorian_mask, "year"] - 621
df["year"] = df["year"].round().astype("Int64")

# Mileage should be numeric.
df["mileage_km"] = pd.to_numeric(df["mileage_km"], errors="coerce")


## 5. Remove exact duplicates


In [27]:
duplicate_count = int(df.duplicated().sum())
print("Duplicate rows found:", duplicate_count)
df = df.drop_duplicates().copy()


Duplicate rows found: 10


## 6. Run basic validation checks

Check that required columns exist, numeric fields have sensible types, and missing values or duplicate rows are visible.


In [28]:
required = [
    "brand", "model", "trim", "year", "mileage_km",
    "city", "price_toman", "published_date", "body_condition"
]
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Shape:", df.shape)
print("Duplicate rows:", int(df.duplicated().sum()))
print("\nMissing values:")
display(df[required].isna().sum().to_frame("missing"))
print("\nNumeric statistics:")
display(df[["year", "mileage_km", "price_toman"]].describe(
    percentiles=[.01, .05, .25, .50, .75, .95, .99]
))


Shape: (4010, 9)
Duplicate rows: 0

Missing values:


,missing
brand,0
model,0
trim,0
year,0
mileage_km,0
city,0
price_toman,0
published_date,2118
body_condition,0



Numeric statistics:


,year,mileage_km,price_toman
count,4010.0,4.010000e+03,4010.000000
mean,1392.660599,1.990957e+05,2936.825162
std,7.658218,1.406690e+05,4559.114688
min,1344.0,0.000000e+00,115.000000
1%,1371.0,0.000000e+00,315.270000
5%,1381.0,1.000000e+00,500.000000
25%,1388.0,9.112500e+04,950.000000
50%,1393.0,1.900000e+05,1520.000000
75%,1398.0,2.870000e+05,3050.000000
95%,1404.0,4.411000e+05,9800.000000


### every thing works well!

## 7. Review market-level sanity checks

In [29]:
print("Top brands:")
display(df["brand"].value_counts().head(10))

print("Top models:")
display(df["model"].value_counts().head(10))

print("Models with fewer than 5 listings:")
model_counts = df.groupby(["brand", "model"]).size().sort_values()
display(model_counts[model_counts < 5].head(20))


Top brands:


brand
پژو         1068
رنو          286
ام وی ام     252
تویوتا       170
کیا          158
هیوندای      149
دنا          136
کوییک        134
سیتروئن      126
مزدا         118
Name: count, dtype: int64

Top models:


model
206             586
206 SD          170
207             152
تندر 90         146
زانتیا          120
پلاس EF7         84
پارس             78
110              64
ریو (مونتاژ)     61
405              60
Name: count, dtype: int64

Models with fewer than 5 listings:


brand               model       
آئودی               Q7              1
آرتابان (ای ام جی)  توکا            1
آریسان              بنزینی          1
گریت وال            وانت دییر       1
آی ام موتورز        LS7             1
بستیون              نات             1
اپل                 کورسا           1
کیا                 اپیروس          1
آئودی               Q4 e-tron       1
چانگان              CS 35 پلاس      1
هیوندای             کرتا            1
                    پالیسید         1
ب ام و              سری 1 هاچ بک    1
                    سری 1 کوپه      1
اینوی               هیبرید          1
اپل                 وکترا           1
                    موکا            1
ام وی ام            X55             1
اپل                 آسترا سدان      1
هیوندای             کوپه            1
dtype: int64

### contrary to our limitations, we have a strong dataset that is well-destributed in many brands and models.
### obviously, our model would be weak on some cars such  as Audi Q7 due to fewer cases!

## 8. removing outliers!
### using IQR which is a basic method for marking outliers. it's not strong and precise but, it would be just OK due to the project's timing circumstance.
### also, I prefered to removing them instead of replacing values or other options. 
#### there are planty of options for treating much better!

In [34]:
group_cols = ["brand", "model", "year"]

stats = (
    df.groupby(group_cols)["price_toman"]
      .agg(
          count="count",
          q1=lambda x: x.quantile(0.25),
          q3=lambda x: x.quantile(0.75)
      )
)

stats["iqr"] = stats["q3"] - stats["q1"]
stats["lower"] = stats["q1"] - 1.5 * stats["iqr"]
stats["upper"] = stats["q3"] + 1.5 * stats["iqr"]

df["possible_outlier"] = False

for keys, group in df.groupby(group_cols):
    row = stats.loc[keys]

    if row["count"] >= 5:
        df.loc[group.index, "possible_outlier"] = (
            (group["price_toman"] < row["lower"]) |
            (group["price_toman"] > row["upper"])
        )

outlier_count = int(df["possible_outlier"].sum())
print("Potential outliers found:", outlier_count)

# removing outliers
df = df.loc[~df["possible_outlier"]].copy()

df.drop(columns=["possible_outlier"], inplace=True)

print("Remaining rows:", len(df))

Potential outliers found: 20
Remaining rows: 3867


## 9. Export the cleaned dataset


In [37]:
print("Final shape:", df.shape)
print("Final duplicates:", int(df.duplicated().sum()))
print("Final missing values:", int(df.isna().sum().sum()))

df.to_csv("cars_cleaned.csv", index=False, encoding="utf-8-sig")
print("Saved: cars_cleaned.csv")


Final shape: (3867, 9)
Final duplicates: 0
Final missing values: 2042
Saved: cars_cleaned.csv
